# Create Demo Table

This notebook creates a demo table in the configured Unity Catalog schema using Databricks Bundle variables.

In [ ]:
# Widget setup for catalog and schema (safe defaults)
dbutils.widgets.text("catalog", "hive_metastore")
dbutils.widgets.text("schema", "default")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"➡️ Using catalog={catalog}, schema={schema}")

In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Get catalog and schema from bundle variables (with safe fallback)
catalog = os.getenv("DATABRICKS_BUNDLE_VAR_catalog", catalog)
schema = os.getenv("DATABRICKS_BUNDLE_VAR_schema", schema)
if not catalog:
    raise ValueError("❌ No catalog provided. Check databricks.yml target overrides.")

table_name = f"{catalog}.{schema}.demo_sales"

print(f"Creating schema (if not exists): {catalog}.{schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

print(f"Creating table: {table_name}")
spark.sql(f"""
CREATE OR REPLACE TABLE {table_name} AS
SELECT 1 AS id, 'productA' AS product, 100 AS amount, 100 * 1.1 AS amount_with_tax
UNION ALL
SELECT 2 AS id, 'productB' AS product, 200 AS amount, 200 * 1.1 AS amount_with_tax
""")

print(f"✅ Table {table_name} created successfully with amount_with_tax column.")

In [ ]:
display(spark.table(table_name))